# Auditoría de calidad: de pilotos y planos a curvas y proyecciones

Este notebook complementa `Proyecciones_teoricas_todos_los_anios.ipynb`. Revisa todas las ediciones 2021–2026, distingue motivos de exclusión y corrige una pérdida verificable: copias de una misma referencia de piloto idénticas en todos los atributos salvo fila e ID técnico. No elimina movimientos de corte, no relaja ciclos ni inventa variedades o edades.

La auditoría parte de las tablas de procedencia guardadas desde Excel. Conserva el estado anterior por observación y tablas antes/después; reconstruye curvas con la misma ventana de 78 semanas, IQR y horizonte de diez semanas. Las fuentes Excel permanecen intactas. Después se debe ejecutar el notebook de todos los años y la cadena analítica para propagar resultados. Los totales de plantas entre planos/horizontes son exposiciones repetidas, nunca inventario físico único.


**Segunda corrección:** identidad comercial única con múltiples IDs del maestro. La clave de curva es producto/color/variedad, sin elegir un ID SQL. Se conserva la lista de códigos y se separan las combinaciones realmente ambiguas. Las cifras antes/después incluyen ambas correcciones.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import gc
from datetime import date
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'Datos_analiticos_proyecto').exists())
SALIDA=ROOT/'Notebooks/Proyecciones Teoricas Propias/resultados_todos_los_anios'
AUD=SALIDA/'auditoria_calidad'; AUD.mkdir(exist_ok=True)
ANIOS=list(range(2021,2027))
VENTANA_CURVA_SEMANAS=78
HORIZONTE=10
def registrar(x):print(x,flush=True)
def normalizar(s):
    return s.astype('string').str.strip().str.upper().str.normalize('NFKD').str.encode('ascii',errors='ignore').str.decode('ascii').str.replace(r'\s+',' ',regex=True).replace('',pd.NA)
def lunes_iso(anio,semana):
    def convertir(a,s):
        try:
            if pd.isna(a) or pd.isna(s) or a!=int(a) or s!=int(s) or not 1980<=a<=2100:return pd.NaT
            return pd.Timestamp(date.fromisocalendar(int(a),int(s),1))
        except ValueError:return pd.NaT
    return pd.Series([convertir(a,s) for a,s in zip(anio,semana)],index=anio.index,dtype='datetime64[ns]')

import numpy as np
import re
from datetime import date
import matplotlib.pyplot as plt

RUTA_MAESTROS = ROOT / 'Notebooks' / 'Extracción SQL' / 'Datos' / '01_maestros_y_geografia'
import json
CATALOGO=RUTA_MAESTROS/'Catalogo_analitico'
assert (CATALOGO/'maestro_variedades_integrado.parquet').exists(), 'Ejecutar primero 00_catalogo_variedades.ipynb'
maestro_productos=pd.read_parquet(CATALOGO/'productos.parquet')
maestro_colores=pd.read_parquet(CATALOGO/'colores.parquet')
maestro=pd.read_parquet(CATALOGO/'maestro_variedades_integrado.parquet')

def normalizar(s):
    return (s.astype('string').str.strip().str.upper().str.normalize('NFKD')
            .str.encode('ascii', errors='ignore').str.decode('ascii')
            .str.replace(r'\s+', ' ', regex=True).replace('', pd.NA))

def vacios(d):
    return d.replace(['','NULL','null','None','NONE','N/A','n/a','NA','na'], np.nan).replace(r'^\s*$', np.nan, regex=True)

def campo(d, *nombres):
    cols = dict(zip(normalizar(pd.Series(d.columns)), d.columns))
    for nombre in normalizar(pd.Series(nombres)):
        if nombre in cols:
            return d[cols[nombre]]
    return pd.Series(pd.NA, index=d.index, dtype='object')

def fechas(s):
    # No interpretar seriales Excel como nanosegundos desde 1970.
    numero = pd.to_numeric(s, errors='coerce')
    f = pd.to_datetime(s.where(numero.isna()), errors='coerce', format='mixed', dayfirst=True)
    serial = numero.between(20000, 80000)
    f.loc[serial] = pd.to_datetime(numero.loc[serial], unit='D', origin='1899-12-30')
    return f.where(f.between('1980-01-01','2100-12-31'))

def lunes_iso(anio, semana):
    # Una semana imposible (p. ej. 53 en ciertos años) se reporta, no se desplaza silenciosamente.
    pares = pd.DataFrame({'a':pd.to_numeric(anio, errors='coerce'), 's':pd.to_numeric(semana, errors='coerce')}).fillna(-1)
    def convertir(t):
        a,s=t
        try:
            if pd.isna(a) or pd.isna(s) or a != int(a) or s != int(s) or not 1980 <= a <= 2100: return pd.NaT
            return pd.Timestamp(date.fromisocalendar(int(a), int(s), 1))
        except ValueError:
            return pd.NaT
    mapa = {t:convertir(t) for t in pares.itertuples(index=False, name=None)}
    return pd.Series([mapa[t] for t in pares.itertuples(index=False, name=None)], index=anio.index, dtype='datetime64[ns]')

equivalencias=json.loads((CATALOGO/'equivalencias.json').read_text(encoding='utf-8'))
ALIAS_PRODUCTOS=equivalencias['productos']
ALIAS_COLORES=equivalencias['colores']
# Rosado y lila no se traducen por intuición: la variedad resuelta aporta su color oficial.
maestro['_var'] = normalizar(maestro.NomVariedad)
maestro['_prod'] = normalizar(maestro.NomProducto)
maestro['_color'] = normalizar(maestro.NomColor)
por_variedad = {k:g for k,g in maestro.groupby('_var')}
def mapa_unico(d, nombre, identificador):
    t=d.assign(k=normalizar(d[nombre]))
    t=t.loc[~t.k.duplicated(keep=False) & t.k.notna()]
    return t.set_index('k')[identificador].to_dict()
ids_producto = mapa_unico(maestro_productos,'NomProducto','IdProducto')
ids_color = mapa_unico(maestro_colores,'NomColor','IdColor')
nom_producto = dict(zip(normalizar(maestro_productos.NomProducto),maestro_productos.NomProducto))
nom_color = dict(zip(normalizar(maestro_colores.NomColor),maestro_colores.NomColor))

def homologar(d):
    d=d.copy()
    for c in ['producto','color','variedad']:d[c+'_original']=d[c]
    claves=pd.DataFrame({'_prod':normalizar(d.producto).replace(ALIAS_PRODUCTOS),
        '_color':normalizar(d.color).replace(ALIAS_COLORES),'_var':normalizar(d.variedad)})
    res=[]
    for prod,col,var in claves.fillna('').drop_duplicates().itertuples(index=False,name=None):
        candidatas=por_variedad.get(var,maestro.iloc[:0])
        if prod:candidatas=candidatas.loc[candidatas._prod.eq(prod)]
        inicial=len(candidatas)
        comerciales=candidatas[['_prod','_color','_var']].drop_duplicates()
        # Diferentes IDs no implican distintas identidades comerciales.
        if len(comerciales)>1 and col:
            candidatas=candidatas.loc[candidatas._color.eq(col)]
            comerciales=candidatas[['_prod','_color','_var']].drop_duplicates()
        r={'_prod':prod,'_color':col,'_var':var,
           'estado_maestro':'sin_coincidencia' if inicial==0 else 'ambiguo',
           'IdVariedad':np.nan,'IdProducto':ids_producto.get(prod,np.nan),'IdColor':ids_color.get(col,np.nan),
           'producto':nom_producto.get(prod,prod or pd.NA),'color':nom_color.get(col,col or pd.NA),
           'variedad':var or pd.NA,'conflicto_color':False,'clave_curva':pd.NA,
           'ids_maestro_equivalentes':'','identidad_comercial_sin_id_unico':False}
        if len(comerciales)==1:
            identidad=comerciales.iloc[0]
            ids=sorted(candidatas.IdVariedad.unique())
            unico=len(ids)==1
            # No escoger arbitrariamente un código SQL cuando existen varios.
            r.update(IdVariedad=ids[0] if unico else np.nan,
                IdProducto=candidatas.IdProducto.iloc[0] if candidatas.IdProducto.nunique()==1 else np.nan,
                IdColor=candidatas.IdColor.iloc[0] if candidatas.IdColor.nunique()==1 else np.nan,
                producto=nom_producto.get(identidad._prod,identidad._prod),
                color=nom_color.get(identidad._color,identidad._color),
                variedad=candidatas.NomVariedad.iloc[0] if unico else identidad._var,
                estado_maestro='unico' if unico else 'comercial_unica_ids_multiples',
                conflicto_color=bool(col and col!=identidad._color),
                clave_curva='COMERCIAL|'+identidad._prod+'|'+identidad._color+'|'+identidad._var,
                ids_maestro_equivalentes=';'.join(str(int(x)) for x in ids),
                identidad_comercial_sin_id_unico=not unico)
        res.append(r)
    m=claves.fillna('').merge(pd.DataFrame(res),on=['_prod','_color','_var'],how='left',validate='many_to_one')
    for c in ['IdVariedad','IdProducto','IdColor','producto','color','variedad','estado_maestro','conflicto_color','clave_curva','ids_maestro_equivalentes','identidad_comercial_sin_id_unico']:
        d[c]=m[c].to_numpy()
    return d


## 1. Reconstrucción conservadora de uniones

Se conserva archivo, variedad homologada y referencia completa. Una fecha de siembra conocida debe coincidir; en su ausencia se exige año/semana cuando están presentes. Sin datos de ciclo sólo se admite una única candidata temporalmente compatible. La única modificación es consultar una vez las copias completamente iguales; discrepancias en plantas, fecha, variedad, sustrato, ubicación u otro atributo siguen siendo ambiguas.


In [2]:
def reconstruir_union(a,p,anio):
    columnas=['archivo','referencia','producto','color','variedad','anio_corte','semana_corte','corte','corte_original','edad_original','producto_original','color_original','variedad_original','IdVariedad','IdProducto','IdColor','estado_maestro','conflicto_color','variedad_llave','referencia_llave','id_estadistica','fecha_corte','clave_curva','ids_maestro_equivalentes','identidad_comercial_sin_id_unico']
    e=a[columnas+['fila_excel_est','fecha_siembra_est','semana_siembra_est','anio_siembra_est','sustrato_est','casa_breeder_est']].copy()
    e=e.rename(columns={c:c.removesuffix('_est') for c in e if c.endswith('_est')})
    df_estadisticas_base=e
    df_camas_piloto=p
    llaves=['archivo','variedad_llave','referencia_llave']
    e=df_estadisticas_base
    p=df_camas_piloto
    completa=e[llaves].notna().all(axis=1)
    # Colapsar sólo copias idénticas en TODOS los atributos, salvo fila e ID técnico.
    # Los registros originales se conservan en camas_piloto; no sumar ni borrar cortes.
    atributos_identidad=[c for c in p.columns if c not in ['id_piloto','fila_excel']]
    pderecha=p.loc[p[llaves].notna().all(axis=1)].drop_duplicates(atributos_identidad)
    campos_ciclo=['fecha_siembra','anio_siembra','semana_siembra']
    candidatas=e.loc[completa,['id_estadistica','fecha_corte','clave_curva','ids_maestro_equivalentes','identidad_comercial_sin_id_unico']+llaves+campos_ciclo].merge(
        pderecha[llaves+campos_ciclo+['id_piloto']],on=llaves,how='inner',suffixes=('_est','_pil'))
    cantidad_inicial=candidatas.groupby('id_estadistica').size()
    tiene_fecha=candidatas.fecha_siembra_est.notna()
    tiene_semana=(candidatas.anio_siembra_est.between(1980,2100) & candidatas.semana_siembra_est.between(1,53))
    compatible_fecha=candidatas.fecha_siembra_est.eq(candidatas.fecha_siembra_pil)
    compatible_semana=(candidatas.anio_siembra_est.eq(candidatas.anio_siembra_pil) &
                        candidatas.semana_siembra_est.eq(candidatas.semana_siembra_pil))
    # No aceptar una siembra posterior a la semana del corte; permite días dentro de esa semana.
    temporal=candidatas.fecha_siembra_pil.isna() | candidatas.fecha_corte.isna() | (
        candidatas.fecha_siembra_pil.le(candidatas.fecha_corte+pd.Timedelta(days=6)))
    compatible=np.where(tiene_fecha,compatible_fecha,np.where(tiene_semana,compatible_semana,True)) & temporal
    candidatas_validas=candidatas.loc[compatible].copy()
    n=candidatas_validas.groupby('id_estadistica').size()
    unicas=candidatas_validas.loc[candidatas_validas.id_estadistica.map(n).eq(1),['id_estadistica','id_piloto']]
    resultado=e.merge(unicas,on='id_estadistica',how='left',validate='one_to_one')
    resultado['candidatas_iniciales']=resultado.id_estadistica.map(cantidad_inicial).fillna(0).astype(int)
    resultado['candidatas_compatibles']=resultado.id_estadistica.map(n).fillna(0).astype(int)
    resultado['estado_union']=np.select([
        ~completa.to_numpy(),resultado.candidatas_iniciales.eq(0),resultado.candidatas_compatibles.eq(0),
        resultado.candidatas_compatibles.gt(1)],
        ['llave_incompleta','sin_pareja','conflicto_ciclo','ambigua'],default='unica')
    atributos=['id_piloto','fila_excel','plantas','fecha_siembra','anio_siembra','semana_siembra','sustrato','casa_breeder']
    resultado=resultado.merge(p[atributos],on='id_piloto',how='left',suffixes=('_est','_pil'),validate='many_to_one')
    for c in ['fecha_siembra','anio_siembra','semana_siembra','sustrato','casa_breeder']:
        resultado['conflicto_'+c]=(resultado[c+'_est'].notna() & resultado[c+'_pil'].notna() &
                                  normalizar(resultado[c+'_est']).ne(normalizar(resultado[c+'_pil'])).fillna(False))
        resultado[c]=resultado[c+'_pil'].combine_first(resultado[c+'_est'])
        resultado['origen_'+c]=np.select([resultado[c+'_pil'].notna(),resultado[c+'_est'].notna()],['piloto','estadistica'],default='faltante')
    resultado['fecha_semana_siembra']=resultado.fecha_siembra-pd.to_timedelta(resultado.fecha_siembra.dt.dayofweek,unit='D')
    resultado['fecha_semana_siembra']=resultado.fecha_semana_siembra.fillna(lunes_iso(resultado.anio_siembra,resultado.semana_siembra))
    resultado['edad']=((resultado.fecha_corte-resultado.fecha_semana_siembra).dt.days//7).astype('Int64')
    resultado['flores_x_planta']=resultado.corte.div(resultado.plantas.where(resultado.plantas.gt(0)))
    resultado['diferencia_edad']=resultado.edad-resultado.edad_original
    llave_corte=['archivo','referencia_llave','variedad_llave','fecha_semana_siembra','fecha_corte']
    resultado['corte_repetido']=resultado.duplicated(llave_corte,keep=False)
    df_estadisticas_completo=resultado
    assert len(resultado)==len(e) and resultado.id_estadistica.is_unique
    reporte_uniones=resultado.groupby(['archivo','estado_union']).agg(filas=('id_estadistica','size'),
        corte_reportado=('corte',lambda s:s.sum(min_count=1))).reset_index()
    reporte_uniones['porcentaje_filas']=100*reporte_uniones.filas/reporte_uniones.groupby('archivo').filas.transform('sum')
    uniones_pendientes=resultado.loc[resultado.estado_union.ne('unica')].copy()
    pilotos_llave_repetida=p.loc[p.duplicated(llaves,keep=False)].sort_values(llaves)


    print('No se eliminan uniones fallidas: revisar uniones_pendientes antes de decidir descartes.')

    r=df_estadisticas_completo
    motivos_eda=pd.DataFrame({
        'union_no_unica':r.estado_union.ne('unica'),'identidad_comercial_sin_resolver':r.clave_curva.isna(),
        'plantas_invalidas':~r.plantas.gt(0),'corte_invalido':~r.corte.ge(0),
        'fecha_corte_invalida':r.fecha_corte.isna(),'edad_invalida':r.edad.isna() | r.edad.lt(0),
        'corte_repetido':r.corte_repetido}).fillna(True)
    control_eda_estadisticas=motivos_eda.sum().rename('filas_afectadas').to_frame()
    control_eda_estadisticas['porcentaje']=100*control_eda_estadisticas.filas_afectadas/len(r)
    df_estadisticas_eda=r.loc[~motivos_eda.any(axis=1)].copy()

    df_estadisticas_eda['anio_edicion'] = int(anio)
    df_estadisticas_completo['anio_edicion'] = int(anio)
    df_camas_piloto['anio_edicion'] = int(anio)
    reporte_uniones['anio_edicion'] = int(anio)
    control_eda_estadisticas = control_eda_estadisticas.reset_index(names='motivo').assign(anio_edicion=int(anio))
    return df_estadisticas_eda, df_estadisticas_completo, df_camas_piloto, reporte_uniones, control_eda_estadisticas


In [3]:
def proyectar(planos, estadisticas):
    est = estadisticas.copy()
    est['_r'] = est.corte / est.plantas
    assert est.plantas.gt(0).all() and est.corte.ge(0).all() and est.edad.ge(0).all()
    assert np.isfinite(est['_r']).all()
    coh = ['archivo', 'anio_plano', 'semana_plano', 'fecha_plano', 'version_seleccionada',
           'finca', 'producto', 'color', 'variedad', 'clave_curva', 'fecha_siembra']
    pp = planos.assign(_pv=planos.plantas.where(planos.plantas.ge(0)))
    cohortes = pp.groupby(coh, dropna=False, as_index=False).agg(
        plantas=('_pv', lambda s: s.sum(min_count=1)), plantas_invalidas=('cantidad_invalida', 'any'))
    salida, auditoria, curvas = [], [], []
    for fecha, c in cohortes.groupby('fecha_plano', sort=True):
        hist = est.loc[est.fecha_corte.ge(fecha - pd.Timedelta(weeks=VENTANA_CURVA_SEMANAS)) & est.fecha_corte.lt(fecha)].copy()
        assert hist.empty or hist.fecha_corte.max() < fecha
        if not hist.empty:
            gh = hist.groupby(['clave_curva', 'edad'])['_r']
            # Cuantiles vectorizados: misma regla IQR que el original, sin un lambda por grupo.
            q = gh.quantile([.25, .75]).unstack().rename(columns={.25: 'q1', .75: 'q3'})
            q['n'] = gh.size()
            hist = hist.join(q, on=['clave_curva', 'edad'], validate='many_to_one')
            li, ls = hist.q1 - 1.5*(hist.q3-hist.q1), hist.q3 + 1.5*(hist.q3-hist.q1)
            atipico = hist.n.ge(4) & (hist['_r'].lt(li) | hist['_r'].gt(ls))
            hist['_r_curva'] = hist['_r'].where(~atipico, hist['_r'].clip(lower=li, upper=ls))
        else:
            atipico = pd.Series(False, index=hist.index)
            hist['_r_curva'] = hist['_r']
        auditoria.append({'anio_plano': int(c.anio_plano.iloc[0]), 'fecha_plano': fecha,
                          'inicio_ventana': fecha - pd.Timedelta(weeks=VENTANA_CURVA_SEMANAS),
                          'max_fecha_corte': hist.fecha_corte.max(), 'observaciones_historicas': len(hist),
                          'puntos_limitados_iqr': int(atipico.sum())})
        cv = hist.groupby(['clave_curva', 'edad'], as_index=False).agg(
            rendimiento_curva=('_r_curva', 'mean'), n_observaciones=('_r', 'size'),
            n_camas=('id_piloto', 'nunique'))
        curvas.append(cv.assign(fecha_plano=fecha, anio_edicion=int(c.anio_plano.iloc[0])))
        ids = set(cv.clave_curva)
        lunes = c.fecha_siembra - pd.to_timedelta(c.fecha_siembra.dt.dayofweek, unit='D')
        for h in range(1, HORIZONTE + 1):
            obj = fecha + pd.Timedelta(weeks=h)
            o = c.copy()
            o['edad'] = ((obj - lunes).dt.days // 7).astype('Int64')
            o = o.merge(cv, on=['clave_curva', 'edad'], how='left', validate='many_to_one')
            o['semana_proyeccion'] = obj
            o['horizonte'] = h
            o['estado'] = np.select([
                o.plantas_invalidas, o.fecha_siembra.isna(), o.edad.lt(0).fillna(False),
                o.clave_curva.isna(), ~o.clave_curva.isin(ids), o.rendimiento_curva.notna()],
                ['plantas_invalidas', 'sin_fecha_siembra', 'siembra_posterior',
                 'sin_variedad_maestra', 'sin_historia_variedad', 'proyectada'], default='edad_fuera_de_curva')
            o['produccion_teorica'] = np.where(o.estado.eq('proyectada'), o.plantas * o.rendimiento_curva,
                                              np.where(o.estado.eq('siembra_posterior'), 0., np.nan))
            salida.append(o)
        registrar(f'Curvas y proyecciones {fecha:%Y-%m-%d}: {len(hist):,} observaciones históricas')
    proy = pd.concat(salida, ignore_index=True).rename(columns={'archivo': 'plano'})
    assert not proy.duplicated(['plano', 'horizonte', 'finca', 'producto', 'color', 'variedad', 'clave_curva', 'fecha_siembra']).any()
    assert proy.groupby('plano').horizonte.nunique().eq(HORIZONTE).all()
    ok = proy.estado.eq('proyectada')
    assert np.allclose(proy.loc[ok, 'produccion_teorica'], proy.loc[ok, 'plantas'] * proy.loc[ok, 'rendimiento_curva'])
    return proy, pd.DataFrame(auditoria), pd.concat(curvas, ignore_index=True)

## 2. Antes/después, verificaciones y reconstrucción anual

La columna `estado_union_antes_copias` permite estudiar el cambio en ejecuciones posteriores. Las observaciones previamente utilizables deben permanecer y conservar corte, plantas, fechas y edad. Se contabilizan las recuperadas que efectivamente entran en alguna ventana histórica de los planos, separándolas de recuperaciones antiguas sin efecto en la proyección.


In [4]:
resumen=[]; motivos=[]; uniones=[]; cobertura=[]; recuperaciones=[]; nombres=[]; soporte=[]; prioridades=[]
for anio in ANIOS:
    carpeta=SALIDA/str(anio)
    a=pd.read_parquet(carpeta/'estadisticas_auditoria.parquet')
    p=pd.read_parquet(carpeta/'camas_piloto.parquet')
    u0=pd.read_parquet(carpeta/'estadisticas.parquet')
    planos=pd.read_parquet(carpeta/'planos_todas_versiones.parquet')
    identidad_ya_aplicada='clave_curva' in a.columns
    def rehacer_identidad(d):
        d=d.copy()
        for col in ['producto','color','variedad']:d[col]=d[col+'_original']
        d=homologar(d)
        if 'variedad_llave' in d:d['variedad_llave']=normalizar(d.variedad)
        return d
    clave_anterior=a.get('clave_curva',pd.Series(pd.NA,index=a.index,dtype='string')).copy()
    clave_planos=planos.get('clave_curva',pd.Series(pd.NA,index=planos.index,dtype='string')).copy()
    a=rehacer_identidad(a); p=rehacer_identidad(p); planos=rehacer_identidad(planos)
    cambio_identidad=not clave_anterior.astype('string').equals(a.clave_curva.astype('string')) or not clave_planos.astype('string').equals(planos.clave_curva.astype('string'))
    firma=[c for c in p if c not in ['fila_excel','id_piloto']]

    anteriores=a.get('estado_union_antes_copias',a.estado_union).copy()
    antes_util=a.get('utilizable_antes_copias',a.id_estadistica.isin(u0.id_estadistica)).copy()
    u,b,_,r,m=reconstruir_union(a,p,anio)
    b['estado_union_antes_copias']=anteriores.to_numpy()
    b['utilizable_antes_copias']=antes_util.to_numpy()
    b['recuperada_copia_identica']=b.estado_union_antes_copias.ne('unica') & b.estado_union.eq('unica')
    u=b.loc[b.id_estadistica.isin(u.id_estadistica)].copy()
    assert len(b)==len(a) and b.id_estadistica.is_unique
    assert b.corte.equals(a.corte)
    assert set(u0.id_estadistica).issubset(set(u.id_estadistica))
    cotejo=u0.merge(u,on='id_estadistica',suffixes=('_antes','_despues'),validate='one_to_one')
    for col in ['corte','plantas','fecha_corte','fecha_siembra','edad']:
        assert cotejo[col+'_antes'].equals(cotejo[col+'_despues'])
    nuevas=u.loc[~u.utilizable_antes_copias].copy()
    fechas=planos.fecha_plano.unique()
    entra=np.zeros(len(nuevas),dtype=bool)
    for fecha in fechas:
        entra |= (nuevas.fecha_corte.ge(fecha-pd.Timedelta(weeks=78)) & nuevas.fecha_corte.lt(fecha)).to_numpy()
    resumen.append({'anio':anio,'estadisticas_fuente':len(a),'pilotos_fuente':len(p),'copias_identicas':int(p.duplicated(firma).sum()),'utilizables_antes':int(antes_util.sum()),'utilizables_despues':len(u),'recuperadas_union':int(b.recuperada_copia_identica.sum()),'recuperadas_utilizables':len(nuevas),'recuperadas_en_ventanas':int(entra.sum()),'utilizables_identidad_comercial_multiple':int(u.identidad_comercial_sin_id_unico.sum())})
    recuperaciones.append(nuevas[['anio_edicion','archivo','fila_excel_est','referencia','variedad','id_estadistica','id_piloto','plantas','corte','fecha_corte','edad']].assign(en_ventana=entra))
    uniones.append(b.groupby(['anio_edicion','archivo','estado_union_antes_copias','estado_union']).size().rename('filas').reset_index())
    motivos.append(m)
    nombres.append(b.loc[b.clave_curva.isna()].groupby(['anio_edicion','producto_original','variedad_original','estado_maestro'],dropna=False).agg(filas=('id_estadistica','size'),corte_reportado=('corte',lambda s:s.sum(min_count=1))).reset_index())
    # Resumen de referencia una sola vez; las ejecuciones siguientes no sobrescriben el antes.
    referencia=AUD/f'cobertura_antes_{anio}.parquet'
    def resumir(q):
        return q.loc[q.version_seleccionada].groupby(['anio_plano','plano','fecha_plano','finca','producto','variedad','horizonte','semana_proyeccion','estado'],dropna=False).agg(cohortes=('estado','size'),plantas=('plantas',lambda s:s.sum(min_count=1)),teoria=('produccion_teorica',lambda s:s.sum(min_count=1))).reset_index()
    if not referencia.exists():resumir(pd.read_parquet(carpeta/'proyecciones_teoricas.parquet')).to_parquet(referencia,index=False)
    # Se recalcula sólo si cambió la vista utilizable; una segunda ejecución recupera resultados.
    cambio=not identidad_ya_aplicada or cambio_identidad or set(u.id_estadistica)!=set(u0.id_estadistica)
    if cambio:
        q,control,curvas=proyectar(planos,u)
        q.to_parquet(carpeta/'proyecciones_teoricas.parquet',index=False)
        curvas.to_parquet(carpeta/'curvas.parquet',index=False)
    else:
        q=pd.read_parquet(carpeta/'proyecciones_teoricas.parquet')
        curvas=pd.read_parquet(carpeta/'curvas.parquet')
    p.to_parquet(carpeta/'camas_piloto.parquet',index=False)
    planos.to_parquet(carpeta/'planos_todas_versiones.parquet',index=False)
    planos.loc[planos.version_seleccionada].to_parquet(carpeta/'planos.parquet',index=False)
    b.to_parquet(carpeta/'estadisticas_auditoria.parquet',index=False)
    u.to_parquet(carpeta/'estadisticas.parquet',index=False)
    previo=pd.read_parquet(referencia).assign(version='antes')
    posterior=resumir(q).assign(version='despues')
    cobertura.extend([previo,posterior])
    soporte.append(curvas.groupby('fecha_plano').agg(puntos_curva=('edad','size'),variedades=('clave_curva','nunique'),puntos_una_observacion=('n_observaciones',lambda s:s.eq(1).sum()),puntos_menos_4=('n_observaciones',lambda s:s.lt(4).sum())).reset_index().assign(anio=anio))
    prioridades.append(q.loc[q.version_seleccionada & ~q.estado.isin(['proyectada','siembra_posterior'])].groupby(['anio_plano','finca','producto','variedad','estado'],dropna=False).agg(cohortes_horizonte=('estado','size'),exposiciones_plantas=('plantas',lambda s:s.sum(min_count=1)),planos=('plano','nunique')).reset_index())
    registrar(f'{anio}: {len(nuevas):,} observaciones utilizables recuperadas; {int(entra.sum()):,} intervienen en alguna ventana')
    del a,p,u0,u,b,q,curvas,planos,cotejo
    gc.collect()
resumen=pd.DataFrame(resumen)
detalle=pd.concat(cobertura,ignore_index=True)
detalle.to_parquet(AUD/'cobertura_por_plano_variedad_horizonte.parquet',index=False)
pd.concat(recuperaciones,ignore_index=True).to_parquet(AUD/'observaciones_recuperadas.parquet',index=False)
uniones=pd.concat(uniones,ignore_index=True); motivos=pd.concat(motivos,ignore_index=True)
nombres=pd.concat(nombres,ignore_index=True); soporte=pd.concat(soporte,ignore_index=True)
prioridades=pd.concat(prioridades,ignore_index=True).sort_values('exposiciones_plantas',ascending=False)
display(resumen)
display(uniones.groupby(['anio_edicion','estado_union']).filas.sum().unstack(fill_value=0))
display(motivos)


No se eliminan uniones fallidas: revisar uniones_pendientes antes de decidir descartes.


2021: 29,211 observaciones utilizables recuperadas; 12,782 intervienen en alguna ventana


No se eliminan uniones fallidas: revisar uniones_pendientes antes de decidir descartes.


2022: 37,123 observaciones utilizables recuperadas; 15,661 intervienen en alguna ventana


No se eliminan uniones fallidas: revisar uniones_pendientes antes de decidir descartes.


...

No se eliminan uniones fallidas: revisar uniones_pendientes antes de decidir descartes.


2026: 74,304 observaciones utilizables recuperadas; 13,963 intervienen en alguna ventana


,anio,estadisticas_fuente,pilotos_fuente,copias_identicas,utilizables_antes,utilizables_despues,recuperadas_union,recuperadas_utilizables,recuperadas_en_ventanas,utilizables_identidad_comercial_multiple
0,2021,249743,12970,2570,176110,205321,30595,29211,12782,5457
1,2022,304076,19250,2582,220422,257545,32892,37123,15661,8565
2,2023,365895,21543,2826,267837,316185,39732,48348,21113,12673
3,2024,432446,24277,3821,319713,379560,44621,59847,25226,18265
4,2025,493956,31189,3636,340391,410583,51580,70192,21737,20825
5,2026,530250,32219,3607,370172,444476,52165,74304,13963,23752


estado_union,ambigua,conflicto_ciclo,llave_incompleta,sin_pareja,unica
anio_edicion,,,,,
2021,5857,975,7109,1281,234521
2022,7530,1319,7769,1925,285533
2023,9312,2661,8486,948,344488
2024,10389,4235,8562,948,408312
2025,38935,3237,11349,1026,439409
2026,39069,3324,11293,1026,475538


,motivo,filas_afectadas,porcentaje,anio_edicion
0,union_no_unica,15222,6.095066,2021
1,identidad_comercial_sin_resolver,23991,9.606275,2021
2,plantas_invalidas,17684,7.080879,2021
3,corte_invalido,800,0.320329,2021
4,fecha_corte_invalida,3,0.001201,2021
5,edad_invalida,11038,4.419743,2021
6,corte_repetido,12750,5.105248,2021
7,union_no_unica,18543,6.098147,2022
8,identidad_comercial_sin_resolver,23159,7.616188,2022
9,plantas_invalidas,20898,6.872624,2022


## 3. Cobertura, prioridades y límites

La tabla de cobertura por plano/variedad/horizonte distingue ausencia de ID maestro, falta de historia, edad sin curva y otras causas. `exposiciones_plantas` suma presencia en planos y horizontes repetidos: sirve para priorizar revisión, no para informar plantas únicas. Las causas de exclusión de estadísticas pueden superponerse; no se suman como pérdidas independientes. Las ediciones también comparten historia: las recuperaciones de años diferentes no son observaciones históricas únicas.

Las variedades sin maestro se dejan con nombre original y estado para revisión. No se autorizan equivalencias por semejanza textual, fechas nuevas ni interpolación. Un punto respaldado por una observación permanece visible; su existencia no demuestra representatividad agronómica.


In [5]:
totales=detalle.groupby(['version','anio_plano','estado'],dropna=False).agg(exposiciones_plantas=('plantas','sum'),cohortes_horizonte=('cohortes','sum')).reset_index()
display(totales)
display(prioridades.loc[prioridades.finca.str.upper().eq('GAITANA')].head(25))
display(nombres.sort_values('filas',ascending=False).head(25))
display(soporte.groupby('anio')[['puntos_curva','puntos_una_observacion','puntos_menos_4']].sum())
with pd.ExcelWriter(AUD/'auditoria_calidad_curvas.xlsx') as writer:
    for nombre,tabla in [('Recuperacion',resumen),('Uniones',uniones),('Exclusiones',motivos),('Cobertura',totales),('Nombres_pendientes',nombres),('Soporte',soporte),('Prioridades',prioridades)]:
        tabla.to_excel(writer,sheet_name=nombre,index=False)
resumen.to_parquet(AUD/'resumen_recuperacion.parquet',index=False)
print('Siguiente paso: todos los años → 10/11/12/13 → diario 01 → 14 → diario 02/03. No mezclar modelos antiguos con proyecciones corregidas.')


,version,anio_plano,estado,exposiciones_plantas,cohortes_horizonte
0,antes,2021,edad_fuera_de_curva,6.231329e+08,185423
1,antes,2021,proyectada,4.587322e+09,1198593
2,antes,2021,siembra_posterior,1.566000e+04,14
3,antes,2021,sin_fecha_siembra,1.200000e+04,10
4,antes,2021,sin_historia_variedad,2.323995e+08,127970
5,antes,2021,sin_variedad_maestra,6.268091e+08,173020
6,antes,2022,edad_fuera_de_curva,5.898602e+08,174888
7,antes,2022,plantas_invalidas,0.000000e+00,20
8,antes,2022,proyectada,4.822008e+09,1064307
9,antes,2022,siembra_posterior,4.870000e+04,30


,anio_plano,finca,producto,variedad,estado,cohortes_horizonte,exposiciones_plantas,planos
1559,2024,GAITANA,Minicarnation,Xue,sin_historia_variedad,8440,58393640.0,52
662,2022,GAITANA,Minicarnation,Dune,sin_historia_variedad,12515,51269590.0,50
1936,2025,GAITANA,Minicarnation,Xue,sin_historia_variedad,11350,50461060.0,52
145,2021,GAITANA,Carnation,JOBIN,sin_variedad_maestra,8180,44728890.0,39
1117,2023,GAITANA,Minicarnation,Academy,edad_fuera_de_curva,4261,42396392.0,52
243,2021,GAITANA,Minicarnation,DOBADOO,sin_variedad_maestra,9190,41979940.0,38
1195,2023,GAITANA,Minicarnation,Xue,sin_historia_variedad,7490,41628980.0,52
1932,2025,GAITANA,Minicarnation,Uchuva,edad_fuera_de_curva,3197,37958702.0,52
261,2021,GAITANA,Minicarnation,LORENZO,edad_fuera_de_curva,4182,35542772.0,52
251,2021,GAITANA,Minicarnation,Dune,sin_historia_variedad,8950,33508630.0,45


,anio_edicion,producto_original,variedad_original,estado_maestro,filas,corte_reportado
453,2025,MINICARNATION,<NA>,sin_coincidencia,6232,1453602.0
541,2026,MINICARNATION,<NA>,sin_coincidencia,6176,1442607.0
333,2024,CARNATION,<NA>,sin_coincidencia,3668,635879.0
510,2026,CARNATION,<NA>,sin_coincidencia,3635,636425.0
421,2025,CARNATION,<NA>,sin_coincidencia,3635,636425.0
238,2023,CARNATION,<NA>,sin_coincidencia,3627,634338.0
154,2022,CARNATION,<NA>,sin_coincidencia,3605,625870.0
364,2024,MINICARNATION,<NA>,sin_coincidencia,3398,783411.0
271,2023,MINICARNATION,<NA>,sin_coincidencia,3395,780903.0
93,2021,MINICARNATION,<NA>,sin_coincidencia,3203,743982.0


,puntos_curva,puntos_una_observacion,puntos_menos_4
anio,,,
2021,461376,166296,262857
2022,443574,135364,207203
2023,509725,154919,237156
2024,618064,195443,325885
2025,539462,166440,257975
2026,282943,69441,105854


Siguiente paso: todos los años → 10/11/12/13 → diario 01 → 14 → diario 02/03. No mezclar modelos antiguos con proyecciones corregidas.


## 4. Catálogo compartido y procedencia

La auditoría, proyecciones y base analítica leen el mismo catálogo de `01_maestros_y_geografia/Catalogo_analitico`. Se comprueba pertenencia de claves y conservación de originales. Las diferencias de código de obtentor quedan expuestas; la agrupación comercial no prueba equivalencia biológica.

In [6]:
import hashlib
huellas=json.loads((CATALOGO/'fuentes_sha256.json').read_text(encoding='utf-8'))
for nombre,esperado in huellas.items():
    assert hashlib.sha256((RUTA_MAESTROS/nombre).read_bytes()).hexdigest()==esperado, 'Maestro cambió: regenerar 00 y toda la cadena'
catalogo=pd.read_parquet(CATALOGO/'identidades_comerciales.parquet')
assert catalogo.clave_curva.is_unique
for anio in ANIOS:
    tabla=pd.read_parquet(SALIDA/str(anio)/'estadisticas_auditoria.parquet')
    assert tabla.clave_curva.dropna().isin(catalogo.clave_curva).all()
    assert tabla.loc[tabla.identidad_comercial_sin_id_unico,'IdVariedad'].isna().all()
    assert tabla.id_estadistica.is_unique
print('Todos los años usan claves del catálogo común; IDs ambiguos no inventados; CSV SQL intactos.')
print('Combinaciones con códigos PickObtentor diferentes:',int(catalogo.revisar_obtentor.sum()),'— revisar agronómicamente; identidad comercial no certifica identidad genética.')


Todos los años usan claves del catálogo común; IDs ambiguos no inventados; CSV SQL intactos.
Combinaciones con códigos PickObtentor diferentes: 33 — revisar agronómicamente; identidad comercial no certifica identidad genética.


## 5. Conservación de plantas y cobertura antes/después

La recuperación cambia cobertura y rendimientos; no cambia las plantas de cada plano/finca/horizonte. Se comprueba cada llave y se informa cobertura ponderada por exposiciones, distinta de la proporción de casos completos del modelo.

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'Datos_analiticos_proyecto').exists())
PROY=ROOT/'Notebooks/Proyecciones Teoricas Propias/resultados_todos_los_anios'
controles=[]
for anio in range(2021,2027):
    antes=pd.read_parquet(PROY/'auditoria_calidad'/f'cobertura_antes_{anio}.parquet')
    despues=pd.read_parquet(PROY/str(anio)/'proyecciones_teoricas.parquet')
    despues=despues.loc[despues.version_seleccionada]
    dims=['plano','finca','horizonte']
    aa=antes.groupby(dims,dropna=False).plantas.sum(); dd=despues.groupby(dims,dropna=False).plantas.sum()
    pd.testing.assert_index_equal(aa.index,dd.index)
    assert np.allclose(aa,dd)
    controles.append({'anio':anio,'plantas_conservadas':True,'cobertura_antes_exposiciones':antes.loc[antes.estado.eq('proyectada'),'plantas'].sum()/antes.plantas.sum(),'cobertura_despues_exposiciones':despues.loc[despues.estado.eq('proyectada'),'plantas'].sum()/despues.plantas.sum()})
conservacion=pd.DataFrame(controles)
display(conservacion)
conservacion.to_excel(PROY/'auditoria_calidad/conservacion_y_cobertura.xlsx',index=False)
print('Se conservan las plantas por cada plano, finca y horizonte. Cobertura ponderada por exposiciones; no sumar como plantas físicas únicas.')


,anio,plantas_conservadas,cobertura_antes_exposiciones,cobertura_despues_exposiciones
0,2021,True,0.755775,0.801544
1,2022,True,0.792245,0.831762
2,2023,True,0.760143,0.820487
3,2024,True,0.806249,0.902034
4,2025,True,0.770384,0.845190
5,2026,True,0.722115,0.788037


Se conservan las plantas por cada plano, finca y horizonte. Cobertura ponderada por exposiciones; no sumar como plantas físicas únicas.
